# 국가법령정보 Open API로 AI 기본법 XML 수집

이 노트북은 법제처 국가법령정보 공동활용 Open API를 사용해 **인공지능 발전과 신뢰 기반 조성 등에 관한 기본법**을 XML로 내려받습니다.

흐름: 법령 목록 조회 → 법령일련번호 확인 → 법령 본문 조회 → 원본 XML 저장 → 조문 단위 미리보기

> API 인증키는 `.env`의 `LAW_OPEN_API_OC`에 저장하세요. 키를 노트북에 직접 입력하거나 커밋하지 마세요.

In [3]:
from pathlib import Path
import os
import re
import xml.etree.ElementTree as ET

import requests
from dotenv import load_dotenv
from IPython.display import display
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_DIR = PROJECT_ROOT / 'data' / 'ai_basic_law'
DATA_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / '.env')
OC = os.getenv('LAW_API_OC')
if not OC:
    raise RuntimeError(".env에 LAW_API_OC=발급받은_인증키 를 설정하세요.")

LIST_API = 'https://www.law.go.kr/DRF/lawSearch.do'
DETAIL_API = 'https://www.law.go.kr/DRF/lawService.do'
LAW_NAME = '인공지능 발전과 신뢰 기반 조성 등에 관한 기본법'
TIMEOUT = (10, 60)


In [4]:
def get_xml(url: str, params: dict) -> bytes:
    response = requests.get(url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    if not response.content.strip():
        raise RuntimeError('API 응답이 비어 있습니다.')
    return response.content

# 현행법령(시행일) 목록 조회: 검색 결과에서 법령일련번호와 상세 링크를 얻습니다.
list_xml = get_xml(LIST_API, {
    'OC': OC,
    'target': 'eflaw',
    'type': 'XML',
    'query': LAW_NAME,
})
(DATA_DIR / 'law_search_result.xml').write_bytes(list_xml)
print(f'목록 XML 저장: {DATA_DIR / "law_search_result.xml"}')
print(list_xml[:200].decode('utf-8', errors='replace'))

목록 XML 저장: /home/aiuser/rag_mini_pjt_template/data/ai_basic_law/law_search_result.xml
<?xml version="1.0" encoding="UTF-8"?><LawSearch><target>eflaw</target><키워드>인공지능 발전과 신뢰 기반 조성 등에 관한 기본법</키워드><section>lawNm</section><totalCnt>7</to


In [5]:
def local_name(tag: str) -> str:
    return tag.rsplit('}', 1)[-1]

def text_of(node, *names):
    for child in node.iter():
        if local_name(child.tag) in names and child.text and child.text.strip():
            return child.text.strip()
    return ''

root = ET.fromstring(list_xml)
records = []
for node in root.iter():
    if local_name(node.tag) not in {'법령정보', 'law'}:
        continue
    law_id = text_of(node, '법령일련번호', '법령ID', '법령id', '법령ID값')
    name = text_of(node, '법령명한글', '법령명')
    link = text_of(node, '법령상세링크', '법령상세링크URL')
    if law_id or name:
        records.append({'law_id': law_id, 'name': name, 'detail_link': link})

# API 응답 구조가 버전에 따라 달라도, 이름이 포함된 항목을 우선 선택합니다.
if not records:
    raise RuntimeError('검색 결과에서 법령 항목을 찾지 못했습니다. law_search_result.xml을 확인하세요.')
matches = [r for r in records if '인공지능' in r['name'] or '신뢰 기반' in r['name']]
selected = matches[0] if matches else records[0]
display(pd.DataFrame(records).drop_duplicates().head(20))
print('선택된 법령:', selected)

,law_id,name,detail_link
0,282791,인공지능 발전과 신뢰 기반 조성 등에 관한 기본법,/DRF/lawService.do?OC=yay&target=eflaw&MST=282...
1,282791,인공지능 발전과 신뢰 기반 조성 등에 관한 기본법,/DRF/lawService.do?OC=yay&target=eflaw&MST=282...
2,276241,인공지능 발전과 신뢰 기반 조성 등에 관한 기본법,/DRF/lawService.do?OC=yay&target=eflaw&MST=276...
3,268543,인공지능 발전과 신뢰 기반 조성 등에 관한 기본법,/DRF/lawService.do?OC=yay&target=eflaw&MST=268...
4,288781,인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령,/DRF/lawService.do?OC=yay&target=eflaw&MST=288...
5,288145,인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령,/DRF/lawService.do?OC=yay&target=eflaw&MST=288...
6,282879,인공지능 발전과 신뢰 기반 조성 등에 관한 기본법 시행령,/DRF/lawService.do?OC=yay&target=eflaw&MST=282...


선택된 법령: {'law_id': '282791', 'name': '인공지능 발전과 신뢰 기반 조성 등에 관한 기본법', 'detail_link': '/DRF/lawService.do?OC=yay&target=eflaw&MST=282791&type=HTML&mobileYn=&efYd=20260721'}


In [6]:
# 본문조회 API: 목록조회에서 얻은 법령일련번호로 전체 본문을 XML로 받습니다.
detail_params = {
    "OC": OC,
    "target": "eflaw",
    "type": "XML",
    "MST": "282791",
    "efYd": "20260721",
}

detail_xml = get_xml(DETAIL_API, detail_params)

xml_path = DATA_DIR / "ai_basic_law.xml"
xml_path.write_bytes(detail_xml)

print(f"본문 XML 저장 완료: {xml_path}")
print(f"크기: {len(detail_xml):,} bytes")
print("법령일련번호: 282791")

본문 XML 저장 완료: /home/aiuser/rag_mini_pjt_template/data/ai_basic_law/ai_basic_law.xml
크기: 145,253 bytes
법령일련번호: 282791


In [7]:
# RAG 전처리 전에 XML 안에 조문이 정상적으로 들어왔는지 간단히 확인합니다.
detail_root = ET.fromstring(detail_xml)
articles = []
for node in detail_root.iter():
    if local_name(node.tag) in {'조문단위', '조문'}:
        article_no = text_of(node, '조문번호', '조문키')
        article_title = text_of(node, '조문제목')
        body = ' '.join(t.strip() for t in node.itertext() if t.strip())
        if body:
            articles.append({
                'article': article_no,
                'title': article_title,
                'text_preview': re.sub(r'\s+', ' ', body)[:200],
            })

articles_df = pd.DataFrame(articles).drop_duplicates()
display(articles_df.head(10))
print(f'찾은 조문 후보 수: {len(articles_df):,}')

,article,title,text_preview
0,1,목적,1 전문 20260721 일부개정 N 제1장 총칙 1 조문 목적 20260721 일...
1,1,,1 전문 20260721 일부개정 N 제1장 총칙
2,1,목적,1 조문 목적 20260721 일부개정 N 제1조(목적) 이 법은 인공지능의 건전한...
3,2,정의,2 조문 정의 20260721 일부개정 Y 제2조(정의) 이 법에서 사용하는 용어의...
4,3,기본원칙 및 국가 등의 책무,3 조문 기본원칙 및 국가 등의 책무 20260721 일부개정 000000 0000...
5,4,적용범위,4 조문 적용범위 20260721 일부개정 N 제4조(적용범위) ① ① 이 법은 국...
6,5,다른 법률과의 관계,5 조문 다른 법률과의 관계 20260721 일부개정 N 제5조(다른 법률과의 관계...
7,6,,6 전문 20260721 일부개정 N 제2장 인공지능의 건전한 발전과 신뢰 기반 조...
8,6,인공지능 기본계획의 수립,6 조문 인공지능 기본계획의 수립 20260721 일부개정 000000 000000...
9,7,국가인공지능전략위원회,7 조문 국가인공지능전략위원회 20260721 일부개정 N 제7조(국가인공지능전략위...


찾은 조문 후보 수: 55


## 다음 단계

저장된 `data/ai_basic_law/ai_basic_law.xml`을 `rag/loader.py`에서 읽고, `장/절/조/항/호/목` 태그를 기준으로 metadata를 붙여 chunking하면 됩니다.

API 사용 시 참고: [국가법령정보 공동활용 Open API 안내](https://open.law.go.kr/LSO/openApi/openApiManual.do)